In [0]:
df = spark.read.format("csv") \
.option("header", True) \
.option("inferSchema", True) \
.load("abfss://projectstorage@eprojectstorageacc.dfs.core.windows.net/bronze/orders/Ecommerce.csv")

df.display()

Create Temp View for SQL Enable

In [0]:
df.createOrReplaceTempView("orders_data")

In [0]:
%sql
--Extracting Customer Data
SELECT
    CustomerID AS customer_id,
    CONCAT_WS(' ', FirstName, LastName) AS customer_name,
    City AS city
FROM orders_data;

In [0]:
#Storing Result in DataFrame
df_customers = spark.sql("""
SELECT 
    CustomerID AS customer_id,
    CONCAT_WS(' ', FirstName, LastName) AS customer_name,
    City AS city
FROM orders_data
""")

df_customers.display()

Setup JDBC Connection

JDBC=Connector

I used JDBC(Java Database Connectivity) to transfer data from Databricks to Azure SQL because it provides a secure and efficient way to write and read data between Spark and relational databases.

“JDBC acts as a bridge between Databricks and SQL database.”

In [0]:
jdbc_url = "jdbc:sqlserver://sqlecommerceserverr.database.windows.net:1433;database=ecommercedatabase"

connection_properties = {
    "user": "project",
    "password": "Sahil@2003",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

Push Data to Azure SQL

In [0]:
df_customers.write.jdbc(
    url=jdbc_url,
    table="customers",
    mode="overwrite",
    properties=connection_properties
)